# IPL Auction — GRPO training (Colab)

This notebook follows the same **install → clone → verify → GRPO → plot** flow as the vaccine cold-chain reference, but targets **this repo**: multi-team IPL auction simulation, **BID/PASS**–style GRPO rewards, and logs under `training/logs/`.

**Live demo (Gradio):** [meta-ipl-hackathon on Hugging Face Spaces](https://huggingface.co/spaces/thirunagarisairamcharan/meta-ipl-hackathon)  
**Training script:** `training/train.py` (GRPO + completion-based rewards; single `trainer.train()` pass).

Use a **GPU runtime** (e.g. T4) for reasonable speed; CPU works but is slow.

In [ ]:
!pip install -q "trl>=0.9.6" "transformers>=4.44.0" "tokenizers>=0.20.0" datasets accelerate torch matplotlib jmespath bitsandbytes openenv
print("Dependencies installed. If you upgraded torch/trl heavily, use: Runtime → Restart session")

### Optional: restart runtime
If Colab warns about version conflicts after `pip install`, restart the session once, then re-run from the **clone** cell downward.

In [ ]:
import os
import subprocess

REPO_URL = "https://huggingface.co/spaces/thirunagarisairamcharan/meta-ipl-hackathon"
WORKDIR = "ipl_meta_hackathon"

if not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
print("cwd:", os.getcwd())
print("top-level:", os.listdir("."))

In [ ]:
# Sanity check: local auction env (no remote Space API required)
import sys
import os

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from env.ipl_env import IPLAuctionEnv

env = IPLAuctionEnv()
obs = env.reset()
# reset() returns {team_id: observation_dict}
print("Reset OK. Team IDs:", list(obs.keys()))
first_team = next(iter(obs))
print(f"Sample obs for {first_team} (keys):", list(obs[first_team].keys())[:12])

In [ ]:
# Run GRPO training (uses training/train.py from the cloned repo)
# Lower --episodes for a quick smoke test (e.g. 5 on CPU)
import os
import subprocess
import sys

EPISODES = 50
r = subprocess.run(
    [sys.executable, "training/train.py", "--episodes", str(EPISODES)],
    cwd=os.getcwd(),
)
if r.returncode:
    raise SystemExit(r.returncode)

In [ ]:
import matplotlib.pyplot as plt
import csv
import os

csv_path = "training/logs/rewards.csv"
if not os.path.isfile(csv_path):
    print("No rewards.csv yet — train first.")
else:
    rows = []
    with open(csv_path, newline="") as f:
        for row in csv.DictReader(f):
            rows.append(row)
    # Expect columns: episode, team_id, TOTAL
    by_ep = {}
    for row in rows:
        try:
            ep = int(row["episode"])
            r = float(row["TOTAL"])
        except (KeyError, ValueError):
            continue
        by_ep.setdefault(ep, []).append(r)
    if by_ep:
        xs = sorted(by_ep.keys())
        ys = [sum(by_ep[x]) / len(by_ep[x]) for x in xs]
        plt.figure(figsize=(8, 4))
        plt.plot(xs, ys, marker="o", color="#004BA0")
        plt.xlabel("Log step / episode index (from CSV)")
        plt.ylabel("Mean logged reward (across teams)")
        plt.title("IPL GRPO — rewards from training/logs/rewards.csv")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig("ipl_reward_curve.png", dpi=150)
        plt.show()
        print("Saved ipl_reward_curve.png")
    else:
        print("CSV had no parseable rows.")

### Next steps
- Inspect `checkpoints/` and `training/logs/reward_events.jsonl` if your clone logs them.
- Point `REPO_URL` at your **GitHub** mirror of the same tree if you prefer `git clone` from GitHub.
- To train **inside the notebook** without subprocess (like the original vaccine notebook), copy the `GRPOTrainer` block from `training/train.py` into a cell after imports.